# 01: Data Preparation

## Objectives
- Download Food-101 dataset
- Extract and organize data
- Select 10 food classes
- Create train/test splits

In [1]:
import os
import zipfile
import tarfile
import requests
from tqdm import tqdm
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
import random

## Configuration

In [2]:
# ==========================================
# CELL 2 : CONFIGURATION
# ==========================================

# Configuration
DATA_DIR = Path('./data')
FOOD101_URL = 'http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz'
FOOD101_TAR = DATA_DIR / 'food-101.tar.gz'
FOOD101_DIR = DATA_DIR / 'food-101'
FOOD10_DIR = DATA_DIR / 'food10'

# Selected 10 classes
SELECTED_CLASSES = [
    'pizza',  'hamburger', 'hot_dog', 'french_fries',
    'ice_cream', 'omelette', 'pancakes', 'ramen', 'steak', 'sushi'
]

# Split configuration
TRAIN_RATIO = 0.75
SEED = 42

# Set random seed
random.seed(SEED)
np.random.seed(SEED)

print(f"Data directory: {DATA_DIR}")
print(f"Selected classes: {len(SELECTED_CLASSES)}")
print(f"Train ratio: {TRAIN_RATIO}")
print(f"Random seed: {SEED}")

Data directory: data
Selected classes: 10
Train ratio: 0.75
Random seed: 42


## Step 1: Download Food-101 Dataset

In [3]:
# ==========================================
# CELL 3 : TÉLÉCHARGER FOOD-101
# ==========================================

def download_food101():
    """Download Food-101 dataset if not exists"""
    if FOOD101_TAR.exists():
        print(f"✅ Dataset already exists: {FOOD101_TAR}")
        return

    print("📥 Downloading Food-101 dataset...")
    print("⏳ Size: ~5GB - This may take 5-10 minutes")
    DATA_DIR.mkdir(exist_ok=True)

    response = requests.get(FOOD101_URL, stream=True)
    total_size = int(response.headers.get('content-length', 0))

    with open(FOOD101_TAR, 'wb') as f, tqdm(
        desc="Downloading",
        total=total_size,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

    print(f"✅ Download completed: {FOOD101_TAR}")

download_food101()

✅ Dataset already exists: data\food-101.tar.gz


In [4]:
# ==========================================
# CELL 4 : EXTRAIRE FOOD-101
# ==========================================

def extract_food101():
    """Extract Food-101 dataset"""
    if FOOD101_DIR.exists():
        print(f"✅ Dataset already extracted: {FOOD101_DIR}")
        return

    print("📦 Extracting Food-101 dataset...")

    try:
        with tarfile.open(FOOD101_TAR, 'r:gz') as tar:
            tar.extractall(DATA_DIR)

        print(f"✅ Extraction completed: {FOOD101_DIR}")

        # Verify extraction
        if FOOD101_DIR.exists():
            print("✅ Extraction successful!")
        else:
            print("❌ Extraction failed - directory not found")

    except Exception as e:
        print(f"❌ Extraction failed: {e}")

extract_food101()

✅ Dataset already extracted: data\food-101


In [5]:
# ==========================================
# CELL 5 : ANALYSER LA STRUCTURE
# ==========================================

def analyze_food101_structure():
    """Analyze the structure of Food-101 dataset"""
    print("\n" + "=" * 60)
    print("📊 FOOD-101 DATASET STRUCTURE")
    print("=" * 60)

    # Check main directories
    images_dir = FOOD101_DIR / 'images'
    meta_dir = FOOD101_DIR / 'meta'

    print(f"Images directory: {images_dir.exists()}")
    print(f"Meta directory: {meta_dir.exists()}")

    if images_dir.exists():
        all_classes = sorted([d.name for d in images_dir.iterdir() if d.is_dir()])
        print(f"\nTotal classes: {len(all_classes)}")
        print(f"First 10 classes: {all_classes[:10]}")

        # Check if our selected classes exist
        print(f"\n✅ Checking selected classes:")
        missing_classes = []
        for cls in SELECTED_CLASSES:
            cls_dir = images_dir / cls
            if cls_dir.exists():
                num_images = len(list(cls_dir.glob('*.jpg')))
                print(f"  ✅ {cls:20s} : {num_images} images")
            else:
                print(f"  ❌ {cls:20s} : NOT FOUND")
                missing_classes.append(cls)

        if missing_classes:
            print(f"\n⚠️ Missing classes: {missing_classes}")
            print("Please update SELECTED_CLASSES with available classes")
        else:
            print(f"\n✅ All selected classes are available!")

    else:
        print("❌ Images directory not found!")

    if not meta_dir.exists():
        print("\n⚠️ Meta directory not found - will create custom splits")

    return images_dir.exists()

dataset_ok = analyze_food101_structure()


📊 FOOD-101 DATASET STRUCTURE
Images directory: True
Meta directory: False

Total classes: 80
First 10 classes: ['apple_pie', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bread_pudding', 'breakfast_burrito', 'bruschetta', 'caesar_salad']

✅ Checking selected classes:
  ✅ pizza                : 1000 images
  ✅ hamburger            : 1000 images
  ✅ hot_dog              : 1000 images
  ✅ french_fries         : 1000 images
  ✅ ice_cream            : 1000 images
  ✅ omelette             : 413 images
  ✅ pancakes             : 1000 images


  ✅ ramen                : 1000 images
  ✅ steak                : 1000 images
  ❌ sushi                : NOT FOUND

⚠️ Missing classes: ['sushi']
Please update SELECTED_CLASSES with available classes

⚠️ Meta directory not found - will create custom splits


In [6]:
# ==========================================
# CELL 6 : CRÉER LES SPLITS PERSONNALISÉS
# ==========================================

def create_custom_splits():
    """Create custom train/test splits from Food-101 images"""
    
    print("\n" + "=" * 60)
    print("📋 CREATING CUSTOM TRAIN/TEST SPLITS")
    print("=" * 60)
    
    images_dir = FOOD101_DIR / 'images'
    
    if not images_dir.exists():
        print(f"❌ Images directory not found: {images_dir}")
        return None, None
    
    train_files = []
    test_files = []
    
    for class_name in SELECTED_CLASSES:
        class_dir = images_dir / class_name
        
        if not class_dir.exists():
            print(f"❌ Class not found: {class_name}")
            continue
        
        # Get all images for this class
        all_images = list(class_dir.glob('*.jpg'))
        
        if len(all_images) == 0:
            print(f"⚠️ No images found for {class_name}")
            continue
        
        # Shuffle
        random.shuffle(all_images)
        
        # Split
        n_train = int(len(all_images) * TRAIN_RATIO)
        
        train_images = all_images[:n_train]
        test_images = all_images[n_train:]
        
        # Add to lists (store relative paths)
        train_files.extend([f"{class_name}/{img.name}" for img in train_images])
        test_files.extend([f"{class_name}/{img.name}" for img in test_images])
        
        print(f"✅ {class_name:20s} : {len(train_images):4d} train, {len(test_images):4d} test")
    
    print(f"\n📊 SPLIT SUMMARY:")
    print(f"  Total train images: {len(train_files)}")
    print(f"  Total test images : {len(test_files)}")
    print(f"  Train ratio       : {len(train_files)/(len(train_files)+len(test_files)):.2%}")
    
    return train_files, test_files

# Create splits
train_files, test_files = create_custom_splits()


📋 CREATING CUSTOM TRAIN/TEST SPLITS
✅ pizza                :  750 train,  250 test
✅ hamburger            :  750 train,  250 test
✅ hot_dog              :  750 train,  250 test
✅ french_fries         :  750 train,  250 test
✅ ice_cream            :  750 train,  250 test
✅ omelette             :  309 train,  104 test
✅ pancakes             :  750 train,  250 test
✅ ramen                :  750 train,  250 test
✅ steak                :  750 train,  250 test
❌ Class not found: sushi

📊 SPLIT SUMMARY:
  Total train images: 6309
  Total test images : 2104
  Train ratio       : 74.99%


In [7]:
# ==========================================
# CELL 7 : CRÉER LA STRUCTURE FOOD-10
# ==========================================

def create_food10_structure():
    """Create Food-10 dataset structure"""
    
    print("\n" + "=" * 60)
    print("📁 CREATING FOOD-10 STRUCTURE")
    print("=" * 60)
    
    # Remove existing directory if it exists
    if FOOD10_DIR.exists():
        print(f"⚠️ Removing existing directory: {FOOD10_DIR}")
        shutil.rmtree(FOOD10_DIR)
    
    # Create directories
    for split in ['train', 'test']:
        for class_name in SELECTED_CLASSES:
            (FOOD10_DIR / split / class_name).mkdir(parents=True, exist_ok=True)
    
    print(f"✅ Created Food-10 structure: {FOOD10_DIR}")
    
    # Show structure
    for split in ['train', 'test']:
        split_dir = FOOD10_DIR / split
        classes = [d.name for d in split_dir.iterdir() if d.is_dir()]
        print(f"  {split}: {len(classes)} classes")

create_food10_structure()


📁 CREATING FOOD-10 STRUCTURE
⚠️ Removing existing directory: data\food10
✅ Created Food-10 structure: data\food10
  train: 10 classes
  test: 10 classes


In [8]:
# ==========================================
# CELL 8 : COPIER LES IMAGES
# ==========================================

def copy_selected_images(train_files, test_files):
    """Copy selected images to Food-10 structure"""
    
    print("\n" + "=" * 60)
    print("📂 COPYING IMAGES TO FOOD-10")
    print("=" * 60)
    
    source_dir = FOOD101_DIR / 'images'
    
    def copy_files(file_list, split_name):
        print(f"\n📥 Copying {split_name} images...")
        copied = 0
        missing = 0
        
        for file_path in tqdm(file_list, desc=f"Copying {split_name}"):
            # Parse class_name/filename.jpg
            parts = file_path.split('/')
            class_name = parts[0]
            filename = parts[1]
            
            source_file = source_dir / class_name / filename
            dest_file = FOOD10_DIR / split_name / class_name / filename
            
            try:
                if source_file.exists():
                    shutil.copy2(source_file, dest_file)
                    copied += 1
                else:
                    missing += 1
                    print(f"\n⚠️ File not found: {source_file}")
            except Exception as e:
                missing += 1
                print(f"\n❌ Error copying {source_file}: {e}")
        
        print(f"✅ Copied: {copied}, Missing: {missing}")
        return copied, missing
    
    if train_files is None or test_files is None:
        print("❌ No files to copy - splits not created")
        return
    
    train_copied, train_missing = copy_files(train_files, 'train')
    test_copied, test_missing = copy_files(test_files, 'test')
    
    print(f"\n📊 COPY SUMMARY:")
    print(f"  Train: {train_copied} copied, {train_missing} missing")
    print(f"  Test : {test_copied} copied, {test_missing} missing")
    
    return train_copied, test_copied

# Copy images
train_copied, test_copied = copy_selected_images(train_files, test_files)


📂 COPYING IMAGES TO FOOD-10

📥 Copying train images...


Copying train:   0%|          | 10/6309 [00:00<01:27, 71.79it/s]

Copying train: 100%|██████████| 6309/6309 [00:23<00:00, 264.63it/s]


✅ Copied: 6309, Missing: 0

📥 Copying test images...


Copying test: 100%|██████████| 2104/2104 [00:06<00:00, 322.67it/s]

✅ Copied: 2104, Missing: 0

📊 COPY SUMMARY:
  Train: 6309 copied, 0 missing
  Test : 2104 copied, 0 missing


In [9]:
# ==========================================
# CELL 9 : VÉRIFIER LE DATASET FOOD-10
# ==========================================

def verify_food10_dataset():
    """Verify the created Food-10 dataset"""
    
    print("\n" + "=" * 60)
    print("✅ FOOD-10 DATASET VERIFICATION")
    print("=" * 60)
    
    dataset_stats = {}
    
    for split in ['train', 'test']:
        split_stats = {}
        total_images = 0
        
        print(f"\n{split.upper()}:")
        
        for class_name in SELECTED_CLASSES:
            class_dir = FOOD10_DIR / split / class_name
            if class_dir.exists():
                images = list(class_dir.glob('*.jpg'))
                count = len(images)
                split_stats[class_name] = count
                total_images += count
                print(f"  {class_name:20s} : {count:4d} images")
            else:
                split_stats[class_name] = 0
                print(f"  {class_name:20s} : 0 images (❌ directory not found)")
        
        dataset_stats[split] = {
            'per_class': split_stats,
            'total': total_images
        }
        
        print(f"  {'─' * 40}")
        print(f"  TOTAL                : {total_images:4d} images")
    
    # Calculate statistics
    print(f"\n" + "=" * 60)
    print("📊 DATASET STATISTICS")
    print("=" * 60)
    print(f"Total images       : {dataset_stats['train']['total'] + dataset_stats['test']['total']}")
    print(f"Train images       : {dataset_stats['train']['total']}")
    print(f"Test images        : {dataset_stats['test']['total']}")
    print(f"Number of classes  : {len(SELECTED_CLASSES)}")
    print(f"Avg per class      : {(dataset_stats['train']['total'] + dataset_stats['test']['total']) / len(SELECTED_CLASSES):.1f}")
    
    # Check for issues
    issues = []
    for split in ['train', 'test']:
        for class_name, count in dataset_stats[split]['per_class'].items():
            if count == 0:
                issues.append(f"{split}/{class_name}: 0 images")
    
    if issues:
        print(f"\n⚠️ ISSUES DETECTED:")
        for issue in issues:
            print(f"  • {issue}")
    else:
        print(f"\n✅ No issues detected - dataset is ready!")
    
    return dataset_stats

# Verify dataset
dataset_stats = verify_food10_dataset()


✅ FOOD-10 DATASET VERIFICATION

TRAIN:
  pizza                :  750 images
  hamburger            :  750 images
  hot_dog              :  750 images
  french_fries         :  750 images
  ice_cream            :  750 images
  omelette             :  309 images
  pancakes             :  750 images
  ramen                :  750 images
  steak                :  750 images
  sushi                :    0 images
  ────────────────────────────────────────
  TOTAL                : 6309 images

TEST:
  pizza                :  250 images
  hamburger            :  250 images
  hot_dog              :  250 images
  french_fries         :  250 images
  ice_cream            :  250 images
  omelette             :  104 images
  pancakes             :  250 images
  ramen                :  250 images
  steak                :  250 images
  sushi                :    0 images
  ────────────────────────────────────────
  TOTAL                : 2104 images

📊 DATASET STATISTICS
Total images       : 8413
Train

In [10]:
def verify_food10_dataset():
    """Verify the created Food-10 dataset"""
    
    print("\nFood-10 Dataset Verification:")
    print("=" * 40)
    
    dataset_stats = {}
    
    for split in ['train', 'test']:
        split_stats = {}
        total_images = 0
        
        for class_name in SELECTED_CLASSES:
            class_dir = FOOD10_DIR / split / class_name
            if class_dir.exists():
                images = list(class_dir.glob('*.jpg'))
                count = len(images)
                split_stats[class_name] = count
                total_images += count
            else:
                split_stats[class_name] = 0
        
        dataset_stats[split] = {
            'per_class': split_stats,
            'total': total_images
        }
        
        print(f"\n{split.upper()}:")
        for class_name, count in split_stats.items():
            print(f"  {class_name}: {count} images")
        print(f"  Total: {total_images} images")
    
    return dataset_stats

dataset_stats = verify_food10_dataset()


Food-10 Dataset Verification:

TRAIN:
  pizza: 750 images
  hamburger: 750 images
  hot_dog: 750 images
  french_fries: 750 images
  ice_cream: 750 images
  omelette: 309 images
  pancakes: 750 images
  ramen: 750 images
  steak: 750 images
  sushi: 0 images
  Total: 6309 images

TEST:
  pizza: 250 images
  hamburger: 250 images
  hot_dog: 250 images
  french_fries: 250 images
  ice_cream: 250 images
  omelette: 104 images
  pancakes: 250 images
  ramen: 250 images
  steak: 250 images
  sushi: 0 images
  Total: 2104 images


In [11]:
# ==========================================
# CELL 11 : RÉSUMÉ FINAL
# ==========================================

print("\n" + "=" * 60)
print("🎉 DATA PREPARATION COMPLETED")
print("=" * 60)

# Summary
print(f"\n✅ COMPLETED STEPS:")
print(f"  1. ✅ Downloaded Food-101 dataset")
print(f"  2. ✅ Extracted dataset")
print(f"  3. ✅ Selected {len(SELECTED_CLASSES)} food classes")
print(f"  4. ✅ Created custom train/test splits")
print(f"  5. ✅ Created Food-10 dataset structure")
print(f"  6. ✅ Copied images to Food-10")
print(f"  7. ✅ Verified dataset")
print(f"  8. ✅ Saved metadata")

print(f"\n📊 DATASET SUMMARY:")
print(f"  Train images   : {dataset_stats['train']['total']:,}")
print(f"  Test images    : {dataset_stats['test']['total']:,}")
print(f"  Total images   : {dataset_stats['train']['total'] + dataset_stats['test']['total']:,}")
print(f"  Classes        : {len(SELECTED_CLASSES)}")

print(f"\n📁 DATASET LOCATION:")
print(f"  {FOOD10_DIR.absolute()}")

print(f"\n🚀 NEXT STEPS:")
print(f"  1. Run 02_Data_Analysis.ipynb for exploratory data analysis")
print(f"  2. Run 03_Model_Training.ipynb for baseline model training")
print(f"  3. Run 04_Model_Optimization.ipynb for hyperparameter tuning")

print("\n" + "=" * 60)


🎉 DATA PREPARATION COMPLETED

✅ COMPLETED STEPS:
  1. ✅ Downloaded Food-101 dataset
  2. ✅ Extracted dataset
  3. ✅ Selected 10 food classes
  4. ✅ Created custom train/test splits
  5. ✅ Created Food-10 dataset structure
  6. ✅ Copied images to Food-10
  7. ✅ Verified dataset
  8. ✅ Saved metadata

📊 DATASET SUMMARY:
  Train images   : 6,309
  Test images    : 2,104
  Total images   : 8,413
  Classes        : 10

📁 DATASET LOCATION:
  c:\Users\dell\Downloads\project ai\project ai\notebooks\data\food10

🚀 NEXT STEPS:
  1. Run 02_Data_Analysis.ipynb for exploratory data analysis
  2. Run 03_Model_Training.ipynb for baseline model training
  3. Run 04_Model_Optimization.ipynb for hyperparameter tuning

